In [14]:
import random
from pathlib import Path

import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from PIL import Image
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, Resize, ToTensor
import numpy as np

In [2]:
SEED = 492
random.seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)

In [3]:
data_path = Path("../../data/ISIC")
image_path = data_path / "ISIC_2024_Training_Input"
ground_truth_path = data_path / "ISIC_2024_Training_GroundTruth.csv"

In [4]:
ground_truth_df = pl.read_csv(ground_truth_path)
ground_truth_df

isic_id,malignant
str,f64
"""ISIC_0015670""",0.0
"""ISIC_0015845""",0.0
"""ISIC_0015864""",0.0
"""ISIC_0015902""",0.0
"""ISIC_0024200""",0.0
…,…
"""ISIC_9999937""",0.0
"""ISIC_9999951""",0.0
"""ISIC_9999960""",0.0


In [5]:
class ISICDataset(Dataset):
    def __init__(self, df: pl.DataFrame):
        self.df = df
        self.transform = Compose([Resize((224, 224)), ToTensor()])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.row(idx, named=True)
        img = Image.open(row["image_path"]).convert("RGB")
        return self.transform(img), torch.tensor(row["malignant"], dtype=torch.float32)

In [6]:
df = pl.read_csv(ground_truth_path).with_columns(
    (pl.lit(str(image_path)) + "/" + pl.col("isic_id").cast(pl.Utf8) + ".jpg").alias("image_path")
)

train_df, val_df = train_test_split(df, test_size=0.1, random_state=SEED, stratify=df["malignant"])

train_malignant = train_df.filter(pl.col("malignant") == 1)
train_benign = train_df.filter(pl.col("malignant") == 0)

sampled_benign = train_benign.sample(n=8000, seed=SEED)
final_train_df = pl.concat([train_malignant, sampled_benign])

train_loader = DataLoader(ISICDataset(final_train_df), batch_size=64, shuffle=True, generator=g)
val_loader = DataLoader(ISICDataset(val_df), batch_size=64, shuffle=False)

In [7]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = torch.nn.Linear(model.fc.in_features, 1)
device = torch.device("cuda")
model = model.to(device)

In [8]:
pos_weight = torch.tensor([len(sampled_benign) / len(train_malignant)]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=3)

In [9]:
num_epochs = 30
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels.unsqueeze(1))
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            val_loss += criterion(outputs, labels.unsqueeze(1)).item()

    scheduler.step(val_loss / len(val_loader))
    print(
        f"Epoch {epoch + 1}/{num_epochs}, "
        f"Train Loss: {running_loss / len(train_loader):.4f}, "
        f"Val Loss: {val_loss / len(val_loader):.4f}"
    )

Epoch 1/30, Train Loss: 1.0018, Val Loss: 0.3406
Epoch 2/30, Train Loss: 0.5502, Val Loss: 0.4477
Epoch 3/30, Train Loss: 0.3145, Val Loss: 0.4534
Epoch 4/30, Train Loss: 0.2450, Val Loss: 0.2206
Epoch 5/30, Train Loss: 0.2324, Val Loss: 0.2633
Epoch 6/30, Train Loss: 0.1477, Val Loss: 0.1801
Epoch 7/30, Train Loss: 0.1047, Val Loss: 0.1386
Epoch 8/30, Train Loss: 0.1307, Val Loss: 0.1862
Epoch 9/30, Train Loss: 0.1784, Val Loss: 0.2468
Epoch 10/30, Train Loss: 0.1084, Val Loss: 0.1394
Epoch 11/30, Train Loss: 0.0507, Val Loss: 0.1746
Epoch 12/30, Train Loss: 0.0368, Val Loss: 0.1666
Epoch 13/30, Train Loss: 0.0213, Val Loss: 0.1599
Epoch 14/30, Train Loss: 0.0159, Val Loss: 0.1526
Epoch 15/30, Train Loss: 0.0166, Val Loss: 0.1656
Epoch 16/30, Train Loss: 0.0204, Val Loss: 0.1765
Epoch 17/30, Train Loss: 0.0110, Val Loss: 0.1739
Epoch 18/30, Train Loss: 0.0110, Val Loss: 0.2121
Epoch 19/30, Train Loss: 0.0123, Val Loss: 0.1746
Epoch 20/30, Train Loss: 0.0118, Val Loss: 0.1795
Epoch 21/

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        all_preds.extend(torch.sigmoid(outputs).squeeze().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [15]:
precisions, recalls, thresholds = precision_recall_curve(all_labels, all_preds)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Optimal threshold: {best_threshold:.4f}")
print(f"Max F1: {best_f1:.4f}")

Optimal threshold: 0.9996
Max F1: 0.1235


In [16]:
preds_binary = [1 if p >= best_threshold else 0 for p in all_preds]
final_f1 = f1_score(all_labels, preds_binary)

print(f"Final F1 at optimal threshold: {final_f1:.4f}")

Final F1 at optimal threshold: 0.1235
